In [1]:
import os
import geopandas as gpd
import numpy as np
import pandas as pd

import sys

# use absolute path here
project_path = "/mnt/School/PhD/AI221/Project/"
sys.path.insert(0, project_path)

from src.data_extraction.utils.constants import data_path

### Normalizing m2 to per density (value / km2)

In [2]:
osm_path = os.path.join(data_path, "urban_osm_metrics_raw.csv")
df_osm = pd.read_csv(osm_path)
df_osm.head()

,city_name,psgc,year,poi_count,large_water_m,small_water_m,building_area_m2,gray_zones_area_m2,green_zones_area_m2
0,Batangas City,401005000,2022,86.0,63093.631236,124197.998664,8.293202e+06,1.496309e+07,1.100982e+07
1,Batangas City,401005000,2023,85.0,63973.789556,124198.963310,8.309308e+06,1.544018e+07,1.106789e+07
2,Batangas City,401005000,2024,78.0,64606.507373,129561.679285,8.351656e+06,1.728678e+07,1.106522e+07
3,Batangas City,401005000,2025,79.0,68793.927401,130540.962357,2.534504e+07,1.883119e+07,2.562899e+07
4,City of Alaminos,105503000,2022,18.0,78755.534822,128941.933865,4.540227e+05,9.022245e+06,1.165701e+07


In [3]:
# Project to a metric CRS (e.g., EPSG:32651 for PH) 
# This is necessary for accurate area and distance calculations
psgc = gpd.read_file(
    os.path.join(data_path,"ph_adm3_municities/PH_Adm3_MuniCities.shp.shp")
).to_crs(epsg=32651)
psgc = psgc[psgc["geo_level"] == "City"]
psgc["area_m2_manual"] = psgc.to_crs("EPSG:32651").geometry.area
psgc["area_km2_manual"] = psgc.to_crs("EPSG:32651").geometry.area / 1e6
psgc = psgc.rename(columns={"adm3_psgc": "psgc"})
psgc.head()

,adm1_psgc,adm2_psgc,psgc,adm3_en,geo_level,len_crs,area_crs,len_km,area_km2,geometry,area_m2_manual,area_km2_manual
4,100000000,102800000,102805000,City of Batac,City,66661,158252391,66,158.0,"POLYGON ((247341.309 2003933.537, 247293.327 2...",1.582524e+08,158.252392
11,100000000,102800000,102812000,City of Laoag,City,53964,110146974,53,110.0,"POLYGON ((248393.247 2016552.78, 248424.831 20...",1.101470e+08,110.146975
28,100000000,102900000,102906000,City of Candon,City,62247,77652664,62,77.0,"POLYGON ((230319.064 1907309.065, 230338.289 1...",7.765266e+07,77.652664
56,100000000,102900000,102934000,City of Vigan,City,25067,24485368,25,24.0,"POLYGON ((221597.447 1945779.016, 221718.087 1...",2.448537e+07,24.485369
70,100000000,103300000,103314000,City of San Fernando,City,54233,99006121,54,99.0,"POLYGON ((225191.401 1841887.86, 225339.01 184...",9.900612e+07,99.006121


In [4]:
merged_df = pd.merge(
    df_osm, psgc[["psgc", "area_m2_manual", "area_km2_manual"]],  on="psgc",
)
merged_df.head()

,city_name,psgc,year,poi_count,large_water_m,small_water_m,building_area_m2,gray_zones_area_m2,green_zones_area_m2,area_m2_manual,area_km2_manual
0,Batangas City,401005000,2022,86.0,63093.631236,124197.998664,8.293202e+06,1.496309e+07,1.100982e+07,2.744363e+08,274.436316
1,Batangas City,401005000,2023,85.0,63973.789556,124198.963310,8.309308e+06,1.544018e+07,1.106789e+07,2.744363e+08,274.436316
2,Batangas City,401005000,2024,78.0,64606.507373,129561.679285,8.351656e+06,1.728678e+07,1.106522e+07,2.744363e+08,274.436316
3,Batangas City,401005000,2025,79.0,68793.927401,130540.962357,2.534504e+07,1.883119e+07,2.562899e+07,2.744363e+08,274.436316
4,City of Alaminos,105503000,2022,18.0,78755.534822,128941.933865,4.540227e+05,9.022245e+06,1.165701e+07,1.620700e+08,162.069977


In [5]:
# densities
merged_df["poi_per_km2"] = (
    merged_df["poi_count"] / merged_df["area_km2_manual"]
)

merged_df["large_water_m_per_km2"] = (
    merged_df["large_water_m"] / merged_df["area_km2_manual"]
)

merged_df["small_water_m_per_km2"] = (
    merged_df["small_water_m"] / merged_df["area_km2_manual"]
)

# coverage ratios
merged_df["building_coverage_ratio"] = (
    merged_df["building_area_m2"] / merged_df["area_m2_manual"]
)

merged_df["gray_zones_coverage_ratio"] = (
    merged_df["gray_zones_area_m2"] / merged_df["area_m2_manual"]
)

merged_df["green_zones_coverage_ratio"] = (
    merged_df["green_zones_area_m2"] / merged_df["area_m2_manual"]
)

merged_df = merged_df.drop(columns=["area_m2_manual", "area_km2_manual"])

merged_df.head()

,city_name,psgc,year,poi_count,large_water_m,small_water_m,building_area_m2,gray_zones_area_m2,green_zones_area_m2,poi_per_km2,large_water_m_per_km2,small_water_m_per_km2,building_coverage_ratio,gray_zones_coverage_ratio,green_zones_coverage_ratio
0,Batangas City,401005000,2022,86.0,63093.631236,124197.998664,8.293202e+06,1.496309e+07,1.100982e+07,0.313370,229.902631,452.556718,0.030219,0.054523,0.040118
1,Batangas City,401005000,2023,85.0,63973.789556,124198.963310,8.309308e+06,1.544018e+07,1.106789e+07,0.309726,233.109781,452.560233,0.030278,0.056261,0.040330
2,Batangas City,401005000,2024,78.0,64606.507373,129561.679285,8.351656e+06,1.728678e+07,1.106522e+07,0.284219,235.415299,472.101073,0.030432,0.062990,0.040320
3,Batangas City,401005000,2025,79.0,68793.927401,130540.962357,2.534504e+07,1.883119e+07,2.562899e+07,0.287863,250.673556,475.669416,0.092353,0.068618,0.093388
4,City of Alaminos,105503000,2022,18.0,78755.534822,128941.933865,4.540227e+05,9.022245e+06,1.165701e+07,0.111063,485.935375,795.594203,0.002801,0.055669,0.071926


In [6]:
output_path = os.path.join(data_path, "urban_osm_metrics.csv")
merged_df.to_csv(output_path, index=False)